# Phase 4b — which source did the work? (Kaggle)

Phase 4 trained on two pair sources at once and improved everything: retrieval
Precision@10 45.77 → 61.15, and generic STS 72.17 → 74.54. The combined run
cannot say **which source caused it**, and the two have very different standing:

- **mined** — 7,197 pairs requiring TF-IDF *and* the Phase 3 encoder to agree.
  Carries real domain knowledge, but was mined using TF-IDF, which is the
  system that still wins this benchmark. So some of the retrieval gain may be
  the model learning to imitate TF-IDF rather than learning the domain.
- **simcse** — 45,864 self-pairs. Cannot encode any domain relation at all, so
  any gain it produces is the anisotropy fix, independent of Swiggy.

Two runs, same config as Phase 4, one source each. **~20 minutes total** — the
mined-only run is short (112 steps).

Same setup as the Phase 4 notebook: **GPU T4 x2**, **Internet On**, and the
`echo-phase4` dataset attached via **+ Add Input**.

## 1. Check settings and the attached dataset

In [ ]:
import glob, os, socket, sys, torch

gpu = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if gpu else 'NONE')
try:
    socket.create_connection(('huggingface.co', 443), timeout=8); net = True
except OSError:
    net = False
print('Internet:', 'on' if net else 'OFF')

PAIRS = next(iter(glob.glob('/kaggle/input/**/phase4_pairs.jsonl', recursive=True)), None)
cfg = next(iter(glob.glob('/kaggle/input/**/encoder/config.json', recursive=True)), None)
ENCODER = os.path.dirname(cfg) if cfg else None
print('PAIRS  :', PAIRS or 'NOT FOUND')
print('ENCODER:', ENCODER or 'NOT FOUND')

if not gpu:  sys.exit('Settings > Accelerator > GPU T4 x2, then re-run.')
if not net:  sys.exit('Settings > Internet > On, then re-run.')
if not (PAIRS and ENCODER):
    sys.exit('Attach the echo-phase4 dataset: + Add Input > Datasets > Your Datasets.')
print('\nall checks passed.')

## 2. Code and install

In [ ]:
%cd /kaggle/working
![ -d Echo ] && (cd Echo && git pull -q) || git clone -q https://github.com/ayn-aval/Echo.git
%cd /kaggle/working/Echo
!git log --oneline -1
!pip install -q -U sentence-transformers accelerate 2>&1 | tail -2

## 3. The two runs

Everything except `--sources` is identical to the Phase 4 run, so any difference
is attributable to the pair source and not to a changed hyperparameter.

In [ ]:
for source in ['mined', 'simcse']:
    print(f'\n{"=" * 70}\n{source}\n{"=" * 70}')
    cmd = (f"python -m src.training.train_domain"
           f" --pairs {PAIRS} --encoder {ENCODER}"
           f" --out /kaggle/working/sbert-{source}-only"
           f" --sources {source}"
           f" --batch-size 64 --epochs 1 --lr 2e-5")
    !{cmd}

## 4. STS for all four models

STS needs no labelled pool, so the generic-performance question is answered here
and now. Phase 3 was 72.17 and the combined Phase 4 model 74.54 — this shows how
that 2.37 splits between the two sources.

Retrieval is the other half and cannot be answered on Kaggle: it needs the local
Postgres, and each new model must be re-pooled and its new candidates judged
before its number means anything.

In [ ]:
import pandas as pd
from eval.sts_eval import evaluate_sts
from src.embeddings.sbert import make_encoder

rows = []
for name, path in [
    ('phase3-baseline', ENCODER),
    ('mined-only',      '/kaggle/working/sbert-mined-only/encoder'),
    ('simcse-only',     '/kaggle/working/sbert-simcse-only/encoder'),
]:
    print(name)
    rows.append(evaluate_sts(make_encoder(path, quiet=True), name))

df = pd.concat(rows, ignore_index=True)
df.to_csv('/kaggle/working/phase4b_ablation.csv', index=False)
wide = df.pivot(index='model', columns='dataset', values='spearman')
order = [c for c in ['STS12','STS13','STS14','STS15','STS16','STS-B','SICK-R','Avg']
         if c in wide.columns]
print('\n' + wide[order].to_string())
print('\nFor reference: phase3 72.17 · phase4 combined 74.54')

## 5. Save both encoders

Then **Save Version → Save & Run All** and download the zip from the Output tab.
Copy the printed STS table back into the chat — that is the finding.

In [ ]:
!cd /kaggle/working && zip -qr phase4b.zip sbert-mined-only sbert-simcse-only phase4b_ablation.csv && ls -lh phase4b.zip